<a href="https://colab.research.google.com/github/saad-imran2891/week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad-imran2891/week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?
Answer:
Scoring/ranking, not classification or clustering. The output isn't a single yes/no per page — it's an ordered list, since review capacity is limited and only the top of the queue gets acted on. A classifier alone would treat every "positive" page as equally urgent, which throws away the ordering that actually drives the decision.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
Answer:
No direct "needs refresh" label exists in the data. I'll build a proxy from observed signals only — e.g. trend_direction =="down" (the starter's own proxy) as a first pass, or a stronger future-window version (prior 90 days of features → decline over the next 30 days) if I move to the warehouse release. Either way, this is a proxy, not ground truth — a page can decline for reasons unrelated to content quality (seasonality, consolidation), so I'll note that limitation explicitly rather than treat the proxy as truth.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*
Answer:
Precision@50 — of the top 50 pages the score surfaces, how many show real evidence of decline/opportunity on closer inspection. This matches how the output is actually used (a reviewer works down a capped list), not how well the model fits all rows. The starter pipeline's own numbers give me something to beat: baseline rules score 0.240, random forest scores 0.740 on this metric.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
df.head()
print("Total rows:", df.shape[0])
print("\nTrend direction counts:")
print(df['trend_direction'].value_counts())
print("\n% of rows with trend_direction == 'down':", round((df['trend_direction']=='down').mean()*100, 1))

(30000, 44)
Total rows: 30000

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

% of rows with trend_direction == 'down': 54.2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
With 54.2% of pages trending down, a single threshold rule can't distinguish urgency. Even restricted to "down" pages, trend_pct ranges from -20% to -100% (mean -58.1%, std 23.5) — and severity doesn't line up cleanly with visibility: two pages both dropped 100%, but one sits at position 6.8 (page one, high-stakes) and the other at position 27.5 (buried, low-stakes). A fixed rule flags both identically; a model can weigh trend_pct alongside avg_position, ctr, and engagement_rate together to rank which of these 16,262 pages actually deserve the top of a reviewer's queue.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
down_pages = df[df['trend_direction'] == 'down']

print("Pages trending down:", len(down_pages))
print("\ntrend_pct distribution among 'down' pages:")
print(down_pages['trend_pct'].describe())

print("\nExample spread — how different 'down' can look:")
print(down_pages[['content_id', 'trend_pct', 'avg_position', 'ctr', 'engagement_rate']].sort_values('trend_pct').head(3))
print(down_pages[['content_id', 'trend_pct', 'avg_position', 'ctr', 'engagement_rate']].sort_values('trend_pct').tail(3))

Pages trending down: 16262

trend_pct distribution among 'down' pages:
count    16262.000000
mean       -58.113830
std         23.488605
min       -100.000000
25%        -75.900000
50%        -55.600000
75%        -38.500000
max        -20.000000
Name: trend_pct, dtype: float64

Example spread — how different 'down' can look:
                 content_id  trend_pct  avg_position   ctr  engagement_rate
18004  content_4a77f09bb322     -100.0           6.8  0.44              0.0
17853  content_bfe827636ec1     -100.0          27.5  0.00              0.0
17846  content_7046d2e5ab19     -100.0           6.6  0.00              0.0
                 content_id  trend_pct  avg_position   ctr  engagement_rate
3792   content_b2580dc3fac0      -20.0           5.4  1.85             2.11
6972   content_bf67a444faef      -20.0          45.5  0.00             0.00
23496  content_8223440cd40c      -20.0          17.9  0.03             2.63


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.